# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varshit6236vbr-netizen/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Contract

- **Unit of analysis:** One row represents one content page for one client on one reporting date, identified by `report_date`, `client_hash_id`, and `content_hash_id`.
- **Time window:** I use March 2026 as the mid-panel development month.
- **Lane:** Ranking/Scoring for content-refresh prioritization.

In [2]:
# Verify the row grain for March 2026

grain_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS distinct_grain_rows
FROM read_parquet({table})
WHERE month = '2026-03'
""").df()

grain_check

NameError: name 'con' is not defined

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Field roles

**Features**
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`

**Label / proxy**
- `needs_refresh` is the working proxy for the ranking task.

**Context / identifiers**
- `report_date`
- `client_hash_id`
- `content_hash_id`
- `month`

**Excluded**
- Future-month outcome information is excluded because it would not be available at the decision moment and could cause leakage.

In [ ]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

print("Five features:", features)
print("Label/proxy: needs_refresh")
print("Development month: 2026-03")

In [ ]:
# Fields used for the ranking/scoring lane

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

label = "needs_refresh"

context = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "month"
]

excluded = {
    "future_month_data": "Excluded because it would not be known at the decision moment."
}

print("Features:", features)
print("Label:", label)
print("Context:", context)
print("Excluded:", excluded)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Query 1 — Verify the row grain

I check whether the stated page/client/report-date grain is unique in March 2026.

In [ ]:
grain_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
        AS distinct_grain_rows
FROM read_parquet({table})
WHERE month = '2026-03'
""").df()

grain_check

### Query 2 — Verify row count and date span

I measure the number of March 2026 rows and the observed reporting-date range.

In [ ]:
count_window = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet({table})
WHERE month = '2026-03'
""").df()

count_window

### Query 3 — Verify data availability

I keep only rows where both Search Console and Analytics availability flags are `TRUE`.

In [ ]:
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet({table})
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
""").df()

availability_check

### Five-feature frame

I build a five-feature frame from March 2026 using rows where both Search Console and Analytics data are available.

In [ ]:
feature_df = con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet({table})
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
""").df()

feature_df.head()

### Feature availability

- **`gsc_impressions`** — available at the decision moment because it is an observed Search Console metric for the selected reporting period.
- **`gsc_clicks`** — available at the decision moment because it is an observed Search Console metric for the selected reporting period.
- **`gsc_avg_position`** — available at the decision moment because it is an observed Search Console metric for the selected reporting period.
- **`ga4_pageviews`** — available at the decision moment because it is an observed Analytics metric for the selected reporting period.
- **`ga4_sessions`** — available at the decision moment because it is an observed Analytics metric for the selected reporting period.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Limitation

One limitation of this slice is that the available Search Console and Analytics signals describe observed page performance, but they do not capture every reason a page may need a content refresh. Therefore, the resulting ranking should be treated as decision-support rather than proof that a page must be refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.